In [1]:
!pip install -q vllm

import subprocess, time, requests, os

MODEL = "JunHowie/Qwen3-8B-GPTQ-Int4"

log = open("/tmp/c1_vllm.log", "w")
server = subprocess.Popen(
    [
        "vllm", "serve", MODEL,
        "--host", "127.0.0.1",
        "--port", "8000",
        "--max-model-len", "4096",
        "--max-num-seqs", "1",
        "--gpu-memory-utilization", "0.90",
        "--enforce-eager",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)

for i in range(60):
    if server.poll() is not None:
        print("Server stopped. Last log lines:")
        print(open("/tmp/c1_vllm.log").read()[-6000:])
        break
    try:
        r = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        if r.ok:
            print("SERVER READY:", r.json())
            break
    except requests.RequestException:
        pass
    time.sleep(10)
else:
    print("Still starting. Recent logs:")
    print(open("/tmp/c1_vllm.log").read()[-4000:])

^C
ERROR: Operation cancelled by user


FileNotFoundError: [Errno 2] No such file or directory: 'vllm'

In [4]:
%pip install vllm

  Using cached vllm-0.32.0-cp38-abi3-manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached huggingface_hub-2.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached fastapi-0.136.3-py3-none-any.whl.metadata (27 kB)
  Using cached prometheus_fastapi_instrumentator-8.1.0-py3-none-any.whl.metadata (13 kB)
  Using cached lm_format_enforcer-0.11.3-py3-none-any.whl.metadata (17 kB)
  Using cached xgrammar-0.2.8-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached mistral_common-1.13.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached compressed_tensors-0.17.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached depyf-0.20.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached cbor2-6.1.5-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached oss_harmony-0.0.11-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
 

In [5]:
import torch, vllm

print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

vLLM: 0.32.0
PyTorch: 2.13.0+cu130
CUDA available: True
GPU count: 2


In [7]:
%pip install --force-reinstall --no-deps torchaudio==2.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.8 MB/s  0:00:00
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0+cu128
    Uninstalling torchaudio-2.11.0+cu128:
      Successfully uninstalled torchaudio-2.11.0+cu128
Note: you may need to restart the kernel to use updated packages.


In [8]:
import torch, torchaudio, vllm

print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())
print("vLLM:", vllm.__version__)

PyTorch: 2.13.0+cu130
TorchAudio: 2.11.0+cu130
CUDA available: True
vLLM: 0.32.0


In [9]:
import subprocess, time, requests

MODEL = "JunHowie/Qwen3-8B-GPTQ-Int4"

with open("/tmp/c1_vllm.log", "w") as log:
    server = subprocess.Popen(
        [
            "vllm", "serve", MODEL,
            "--host", "127.0.0.1",
            "--port", "8000",
            "--tensor-parallel-size", "2",
            "--max-model-len", "2048",
            "--max-num-seqs", "1",
            "--gpu-memory-utilization", "0.90",
            "--enforce-eager",
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

print("Starting model server...", flush=True)

for i in range(90):
    if server.poll() is not None:
        print("Server exited. Recent log:")
        print(open("/tmp/c1_vllm.log").read()[-8000:])
        break
    try:
        r = requests.get("http://127.0.0.1:8000/v1/models", timeout=3)
        if r.ok:
            print("SERVER READY:", r.json(), flush=True)
            break
    except requests.RequestException:
        pass
    if i % 6 == 0:
        print(f"Still loading ({i * 5}s)...", flush=True)
    time.sleep(5)
else:
    print("Server is still loading. Recent log:")
    print(open("/tmp/c1_vllm.log").read()[-6000:])

Starting model server...
Still loading (0s)...
Still loading (30s)...
Still loading (60s)...
Still loading (90s)...
Still loading (120s)...
SERVER READY: {'object': 'list', 'data': [{'id': 'JunHowie/Qwen3-8B-GPTQ-Int4', 'object': 'model', 'created': 1793210809, 'owned_by': 'vllm', 'root': 'JunHowie/Qwen3-8B-GPTQ-Int4', 'parent': None, 'max_model_len': 2048, 'permission': [{'id': 'modelperm-a0efd4c0cad419cc', 'object': 'model_permission', 'created': 1793210809, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [19]:
import requests
import json

MODEL = "JunHowie/Qwen3-8B-GPTQ-Int4"
URL = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": 'Return exactly this JSON object: {"status":"ok","test":"c1_generator_live"}'
        }
    ],
    "temperature": 0,
    "max_tokens": 200,
    "response_format": {"type": "json_object"},
    "chat_template_kwargs": {"enable_thinking": False}
}

response = requests.post(URL, json=payload, timeout=120)
print("HTTP:", response.status_code)
response.raise_for_status()

data = response.json()
text = data["choices"][0]["message"]["content"]

print("Response:", repr(text))
print("Usage:", data.get("usage"))

parsed = json.loads(text)
assert parsed == {"status": "ok", "test": "c1_generator_live"}

print("\nPASS: vLLM returned valid JSON with thinking disabled.")

HTTP: 200
Response: '{"status":"ok","test":"c1_generator_live"}'
Usage: {'prompt_tokens': 30, 'total_tokens': 43, 'completion_tokens': 13, 'prompt_tokens_details': None, 'completion_tokens_details': None}

PASS: vLLM returned valid JSON with thinking disabled.


In [20]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.


In [22]:
from pathlib import Path
import shutil
import sys

WORK = Path("/kaggle/working/c1_live_test")

# Recreate the project's package layout
for package, files in {
    "llm": ["generate.py"],
    "eval": ["safety.py", "agent_states.py"],
}.items():
    package_dir = WORK / package
    package_dir.mkdir(parents=True, exist_ok=True)
    (package_dir / "__init__.py").touch()

    for filename in files:
        shutil.copy2(WORK / filename, package_dir / filename)

sys.path.insert(0, str(WORK))

print("Package structure created:")
for p in sorted(WORK.rglob("*.py")):
    print(p.relative_to(WORK))

Package structure created:
agent_states.py
eval/__init__.py
eval/agent_states.py
eval/safety.py
generate.py
llm/__init__.py
llm/generate.py
safety.py
test_agent_states.py
test_generator_contract.py
test_safety_eval.py


In [23]:
import pytest
from pathlib import Path

WORK = Path("/kaggle/working/c1_live_test")

exit_code = pytest.main([
    "-q",
    str(WORK / "test_generator_contract.py"),
])

print("\nPytest exit code:", exit_code)

.................                                                        [100%]
=============================== warnings summary ===============================
c1_live_test/test_generator_contract.py: 17 warnings
  /usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
    return datetime.utcnow().replace(tzinfo=utc)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
17 passed, 17 warnings in 0.04s


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Pytest exit code: 0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [36]:
import json
from llm.generate import ChatCompletionsGenerator, json_object

generator = ChatCompletionsGenerator(
    base_url="http://127.0.0.1:8000/v1",
    model="JunHowie/Qwen3-8B-GPTQ-Int4",
    backend="vllm",
    revision="JunHowie/Qwen3-8B-GPTQ-Int4",
    chat_template_kwargs={"enable_thinking": False},
    response_format={"type": "json_object"},
)

result = generator.generate(
    'Return exactly this JSON object: {"status":"ok","test":"c1_generator_live"}',
    max_tokens=150,
)

print("Backend:", result.backend)
print("Model:", result.model)
print("Prompt tokens:", result.prompt_tokens)
print("Completion tokens:", result.completion_tokens)
print("Latency (ms):", round(result.latency_ms, 2))
print("Response:", repr(result.text))

parsed = json_object(result.text)
assert parsed == {"status": "ok", "test": "c1_generator_live"}
assert result.prompt_tokens > 0
assert result.completion_tokens > 0

print("\nPASS: Live C1 generator integration.")

Backend: vllm
Model: JunHowie/Qwen3-8B-GPTQ-Int4
Prompt tokens: 30
Completion tokens: 13
Latency (ms): 1214.16
Response: '{"status":"ok","test":"c1_generator_live"}'

PASS: Live C1 generator integration.
